In [2]:
import glob

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

In [3]:
def load_clean(pat):
    return pd.concat([
        pd.read_csv(
            name,
            dtype=dict(
                VendorID=int,
                passenger_count=float,
                trip_distance=float,
                RatecodeID="category",
                store_and_fwd_flag=str,
                PULocationID=int, # "category",
                DOLocationID=int, # "category",
                payment_type=int,
                fare_amount=float,
                extra=float,
                mta_tax=float,
                tip_amount=float,
                tolls_amount=float,
                improvement_surcharge=float,
                total_amount=float,
                congestion_surcharge=float,
                Airport_fee=float,
            ),
            parse_dates=[
                "tpep_pickup_datetime",
                "tpep_dropoff_datetime",
            ],
        )
        for name in sorted(glob.glob(f"../dataset-clean/{pat}"))
    ])

rides_clean = load_clean("rides_sample.csv").assign(
    pickup_hour=lambda x: x["tpep_pickup_datetime"].dt.floor("1h"),
    pickup_time_minute=lambda x:
        x["tpep_pickup_datetime"].dt.hour * 60 +
        x["tpep_pickup_datetime"].dt.minute,
    fare_per_hour=lambda x:
        x["fare_amount"] / x["duration_hours"],
)

In [4]:
def group_zone_hour(
    df,
    aggs,
):
    """
    Aggregrate trips into zone-hour statistics

    df : pd.DataFrame
        Trip-level dataframe
    aggs : function, str, list, dict or None
        Argument to pd.DataFrameGroupBy.agg, specify which columsn and
        how aggregation should be done

    Returns:
    pd.DataFrame
        Dataframe with zone-hour and aggregated values
    """

    df = (
        df
            .groupby(["PULocationID", "pickup_hour"], observed=True)
            .agg(aggs)
            .reset_index()
    )
    df.columns = [(f"{x}_{y}" if y else x) for (x, y) in df.columns]

    return df

rides_agg = group_zone_hour(
    rides_clean,
    dict(
        duration_hours=["sum", "count"],
        trip_distance="sum",
        fare_amount="sum",
        total_amount="sum",
        fare_per_hour="mean",
    ),
)

In [6]:
def with_lags(
    df,
    feature,
    agg,
    window="2h",
    offsets=["1h"],
):
    agged = df.groupby("PULocationID")[["pickup_hour", feature]].rolling(
        window,
        on="pickup_hour",
    ).agg(agg)

    for offset in offsets:
        df = pd.merge(
            df,
            agged.assign(
                pickup_hour=agged["pickup_hour"] +
                    (offset if not isinstance(offset, str) else pd.Timedelta(offset)),
            ),
            how="left",
            on=["PULocationID", "pickup_hour"],
            suffixes=("", f"-{agg}-{window}-{offset}"),
        )

    return df

rides_lag = rides_agg

for feat in ["duration_hours_sum", "fare_amount_sum", "total_amount_sum", "fare_per_hour_mean"]:
    for (window, offsets) in [
        ("1h", ["1h", "2h", "3h"]),
        ("24h", ["1d", "2d", "3d", "7d"]),
    ]:
        rides_lag = with_lags(
            rides_lag,
            feat,
            "mean",
            window=window,
            offsets=offsets,
        )

rides_lag.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74988 entries, 0 to 74987
Data columns (total 36 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   PULocationID                    74988 non-null  int64         
 1   pickup_hour                     74988 non-null  datetime64[ns]
 2   duration_hours_sum              74988 non-null  float64       
 3   duration_hours_count            74988 non-null  int64         
 4   trip_distance_sum               74988 non-null  float64       
 5   fare_amount_sum                 74988 non-null  float64       
 6   total_amount_sum                74988 non-null  float64       
 7   fare_per_hour_mean              74988 non-null  float64       
 8   duration_hours_sum-mean-1h-1h   69191 non-null  float64       
 9   duration_hours_sum-mean-1h-2h   68219 non-null  float64       
 10  duration_hours_sum-mean-1h-3h   67144 non-null  float64       
 11  du

In [7]:
rides_lag.to_csv("../dataset-features/rides_sample.csv")